In [ ]:
import gc
import sys
sys.path.append("/home/david/source/psilocybin-eeg")

from pathlib import Path
import pandas as pd
import numpy as np
import mne
from mne_icalabel.iclabel import iclabel_label_components
import seaborn as sns
import matplotlib.pyplot as plt

from src.data.dataset_handler import DatasetHandler
from src.data.dataset_preprocessing import DatasetPreprocessor
from src.definitions.fields import (
    SingleDataMetadata, 
    ChannelTypes, 
    ExperimentNames, 
    CoordinateSystems, 
    PreprocessedDataVariants, 
    ICLabelComponentsClasses, 
    ExcludedICsMetadata, 
    MusicTypeVariants, 
    ConditionVariants,
)
from src.definitions.constants import ProjectPaths

%matplotlib qt
# %matplotlib inline

In [ ]:
dataset_handler = DatasetHandler(ExperimentNames.PSILO_MUSIC, CoordinateSystems.HYDROGEL_257_NO_FIDUCIALS)
dataset_metadata = dataset_handler.dataset_metadata
dataset_metadata = pd.read_csv("/home/david/source/psilocybin-eeg/dataset_metadata.csv")
excluded_ics_metadata_old = pd.read_csv("/home/david/source/psilocybin-eeg/excluded_ics_mapping.csv")
excluded_ics_metadata_new = pd.read_csv("/home/david/source/psilocybin-eeg/excluded_ics_mapping-new.csv")

In [ ]:
dataset_handler.preprocess_all_dataset()

# Check Excluded ICs statistics

In [ ]:
def process_excluded_metadata(data:pd.DataFrame) -> pd.DataFrame:
    # Assuming your dataframe is called df
    result = data.groupby('original_filename').agg({
        'original_filename': 'size',  # Count rows per filename
        'total_ics': 'first',     # Get the total (same across filename)
    }).rename(columns={'original_filename': 'count'})

    result['percentage'] = result['count'] / result["total_ics"]
    percentages = pd.Series(result['percentage'].unique())
    print(percentages.describe())
    
    return result

In [ ]:
def plot_ics_distribution(df: pd.DataFrame):

    plt.figure(figsize=(10, 6))
    sns.histplot(df['percentage'].unique(), bins=20, kde=True)
    plt.xlabel('Percentage (%)')
    plt.ylabel('Frequency')
    plt.title('Distribution of Unique Percentage Values')
    plt.tight_layout()
    plt.show()

In [ ]:
old_ics = process_excluded_metadata(excluded_ics_metadata_old)
new_ics = process_excluded_metadata(excluded_ics_metadata_new)

In [ ]:
def plot_ics_distribution_comparison(df1: pd.DataFrame, df2: pd.DataFrame, 
                                      label1: str = 'Dataset 1', 
                                      label2: str = 'Dataset 2'):
    # Combine unique percentages from both dataframes
    import pandas as pd
    
    data = pd.DataFrame({
        'percentage': list(df1['percentage'].unique()) + list(df2['percentage'].unique()),
        'dataset': [label1] * len(df1['percentage'].unique()) + 
                   [label2] * len(df2['percentage'].unique())
    })
    
    plt.figure(figsize=(10, 6))
    sns.histplot(data=data, x='percentage', hue='dataset', bins=20, 
                 kde=True, alpha=0.5, stat='density')
    plt.xlabel('Percentage (%)')
    plt.ylabel('Density')
    plt.title('Distribution Comparison of Percentage Values')
    plt.legend(title='Dataset')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_ics_distribution_comparison(old_ics, new_ics, "old", "new")

# Selection of Condition for analysis

In [ ]:
# Setup of the processing
participant_id = 37
music_type_enum = MusicTypeVariants.PSYTRANCE
condition_enum = ConditionVariants.PLACEBO

# Basic setup
music_mapping = {
    MusicTypeVariants.CLASSICAL: "MusicTypeVariants.CLASSICAL",
    MusicTypeVariants.PSYTRANCE: "MusicTypeVariants.PSYTRANCE"
}
condition_mapping = {
    ConditionVariants.PLACEBO: "ConditionVariants.PLACEBO",
    ConditionVariants.PSILOCYBIN: "ConditionVariants.PSILOCYBIN"
}

music_type = music_mapping[music_type_enum]
condition = condition_mapping[condition_enum]

df = dataset_metadata[dataset_metadata["SingleDataMetadata.PARTICIPANT_ID"] == participant_id]
df = df[df["SingleDataMetadata.MUSIC_TYPE"] == music_type]
df = df[df["SingleDataMetadata.CONDITION"] == condition]
print(df["SingleDataMetadata.EEG_CONDITION_ID"])
original_filename = df["SingleDataMetadata.FILENAME"].iloc[0]

keys_of_interest = [
    ExcludedICsMetadata.ORIGINAL_FILENAME.value, 
    ExcludedICsMetadata.IC_ID.value, 
    ExcludedICsMetadata.IC_CATEGORY.value,
    ExcludedICsMetadata.MAIN_PROBABILITY.value,
    ExcludedICsMetadata.TOTAL_ICS.value,
]
df = excluded_ics_metadata_new[keys_of_interest]
excluded_df = df[df[ExcludedICsMetadata.ORIGINAL_FILENAME.value] == original_filename]


# Dataseries Analysis

In [ ]:
try:
    del before_ica, after_ica
    gc.collect()
except NameError:
    pass

before_ica = dataset_handler.load_data_file(original_filename, is_processed=True, processed_data_type=PreprocessedDataVariants.RAW_BEFORE_ICA)
after_ica = dataset_handler.load_data_file(original_filename, is_processed=True, processed_data_type=PreprocessedDataVariants.RAW_AFTER_ICA)
ica = dataset_handler.load_data_file(original_filename, is_processed=True, processed_data_type=PreprocessedDataVariants.ICA_COMPONENTS)

### Plot Before ICA

In [ ]:
before_ica.plot(n_channels=50)

### Plot After ICA

In [ ]:
after_ica.plot(n_channels=50)

# Excluded ICs processing

In [ ]:
excluded_df[[ExcludedICsMetadata.IC_ID.value, ExcludedICsMetadata.IC_CATEGORY.value, ExcludedICsMetadata.MAIN_PROBABILITY.value, ExcludedICsMetadata.TOTAL_ICS.value]]

### Plot Excluded IC

In [ ]:
excluded_ic_idx = 1
df = excluded_df[excluded_df[ExcludedICsMetadata.IC_ID.value] == excluded_ic_idx]
ica.copy().plot_sources(
    dataset_handler.dataset_preprocessor.remove_bad_epoch_annotations(
        before_ica.pick_types(eeg=True)
        ), 
    picks=[excluded_ic_idx]
)
df[[ExcludedICsMetadata.IC_ID.value, ExcludedICsMetadata.IC_CATEGORY.value, ExcludedICsMetadata.MAIN_PROBABILITY.value, ExcludedICsMetadata.TOTAL_ICS.value]]

In [ ]:
excluded_ic_idx = 1

try:
    del before_ica, after_ica
    gc.collect()
except NameError:
    pass

try:
    del excluded_data
    gc.collect()
except NameError:
    pass

df = excluded_df[excluded_df[ExcludedICsMetadata.IC_ID.value] == excluded_ic_idx]
original_filename = df[ExcludedICsMetadata.ORIGINAL_FILENAME.value].iloc[0]
excluded_data = dataset_handler.load_excluded_ic_dataseries(original_filename, excluded_ic_idx)
ic_probabilities = dataset_handler.load_data_file(original_filename, is_processed=True, processed_data_type=PreprocessedDataVariants.IC_PROBABILITIES)

print(f"Label of the IC: {df[ExcludedICsMetadata.IC_CATEGORY.value]}")
for i, label in enumerate(dataset_handler.dataset_preprocessor.ic_label_classes_order):
    print(f"{label}: {ic_probabilities[excluded_ic_idx, i]:.4f}")
    
excluded_data.plot(n_channels=50)


### Plot sensor distribution

In [ ]:
excluded_data.plot_sensors(show_names=True)